# Planet Ruler Benchmark Analysis

Interactive visualization and analysis of performance benchmark results.

**Note:** Run this notebook from the project root directory:
```bash
jupyter notebook benchmarks/visualize.ipynb
```

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# these are here for developers
%load_ext autoreload
%autoreload 2

# Import benchmark analyzer
from planet_ruler.benchmarks.analyzer import BenchmarkAnalyzer

In [ ]:
# Configure plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load analyzer
analyzer = BenchmarkAnalyzer()

print("✓ Setup complete")

## Load Results

In [ ]:
# Load all results
# df = analyzer.get_results()
df = analyzer.explode_parameters()

print(f"Total benchmark runs: {len(df)}")
print(f"Unique scenarios: {df['scenario_name'].nunique()}")
print(f"Unique images: {df['image_name'].nunique()}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")

df.head()

## Summary Statistics

In [ ]:
summary = analyzer.get_summary_stats()
summary

## Performance vs Accuracy Trade-off

Visualize the Pareto frontier of runtime vs accuracy.

In [ ]:
# Filter successful runs
success_df = df[df["convergence_status"] == "success"].copy()

fig, ax = plt.subplots(figsize=(12, 8))

# Plot all scenarios
for scenario in success_df["scenario_name"].unique():
    scenario_df = success_df[success_df["scenario_name"] == scenario]
    ax.scatter(
        scenario_df["total_time"],
        scenario_df["relative_error"] * 100,
        label=scenario,
        alpha=0.6,
        s=100,
    )

# Add reference lines
ax.axhline(y=20, color="r", linestyle="--", alpha=0.3, label="20% error threshold")
ax.axvline(x=15, color="g", linestyle="--", alpha=0.3, label="15s mobile target")

ax.set_xlabel("Runtime (seconds)", fontsize=12)
ax.set_ylabel("Relative Error (%)", fontsize=12)
ax.set_title("Performance vs Accuracy Trade-off", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Pareto Frontier

Identify the optimal configurations.

In [ ]:
pareto_df = analyzer.get_pareto_frontier()

print(f"Pareto-optimal configurations: {len(pareto_df)}\n")

pareto_df[
    ["scenario_name", "image_name", "total_time", "relative_error", "iterations"]
].sort_values("total_time")

## Runtime Distribution by Scenario

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Box plot of runtimes
success_df.boxplot(column="total_time", by="scenario_name", ax=ax, rot=45)

ax.axhline(y=15, color="r", linestyle="--", alpha=0.5, label="Mobile target (15s)")
ax.set_xlabel("Scenario", fontsize=12)
ax.set_ylabel("Runtime (seconds)", fontsize=12)
ax.set_title("Runtime Distribution by Scenario", fontsize=14, fontweight="bold")
plt.suptitle("")  # Remove auto-generated title
ax.legend()

plt.tight_layout()
plt.show()

## Timing Breakdown

Identify bottlenecks in each scenario.

In [ ]:
timing_breakdown = analyzer.get_timing_breakdown()
timing_breakdown

In [ ]:
# Stacked bar chart of timing phases
fig, ax = plt.subplots(figsize=(14, 6))

scenarios = success_df["scenario_name"].unique()
phases = ["image_load_time", "detection_time", "optimization_time"]
phase_labels = ["Image Load", "Detection", "Optimization"]

mean_times = success_df.groupby("scenario_name")[phases].mean()

mean_times.plot(
    kind="bar", stacked=True, ax=ax, color=["#1f77b4", "#ff7f0e", "#2ca02c"]
)

ax.set_xlabel("Scenario", fontsize=12)
ax.set_ylabel("Time (seconds)", fontsize=12)
ax.set_title("Timing Breakdown by Phase", fontsize=14, fontweight="bold")
ax.legend(phase_labels, title="Phase")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

## Convergence Analysis

In [ ]:
# Iterations vs error
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Iterations vs runtime
for scenario in success_df["scenario_name"].unique():
    scenario_df = success_df[success_df["scenario_name"] == scenario]
    ax1.scatter(
        scenario_df["iterations"], scenario_df["total_time"], label=scenario, alpha=0.6
    )

ax1.set_xlabel("Iterations", fontsize=12)
ax1.set_ylabel("Runtime (seconds)", fontsize=12)
ax1.set_title("Iterations vs Runtime", fontsize=13, fontweight="bold")
ax1.grid(True, alpha=0.3)

# Iterations vs error
for scenario in success_df["scenario_name"].unique():
    scenario_df = success_df[success_df["scenario_name"] == scenario]
    ax2.scatter(
        scenario_df["iterations"],
        scenario_df["relative_error"] * 100,
        label=scenario,
        alpha=0.6,
    )

ax2.set_xlabel("Iterations", fontsize=12)
ax2.set_ylabel("Relative Error (%)", fontsize=12)
ax2.set_title("Iterations vs Accuracy", fontsize=13, fontweight="bold")
ax2.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Compare Specific Scenarios

In [ ]:
# Select scenarios to compare (edit as needed)
scenarios_to_compare = success_df["scenario_name"].unique()[:5]

comparison = analyzer.compare_scenarios(list(scenarios_to_compare), metric="total_time")

print("Runtime comparison (seconds):")
comparison

## Identify Bottlenecks

Find which phases consume the most time.

In [ ]:
# Analyze bottlenecks for each scenario
for scenario in success_df["scenario_name"].unique():
    bottlenecks = analyzer.identify_bottlenecks(scenario, threshold=0.3)
    if bottlenecks:
        print(f"\n{scenario}:")
        for phase, fraction in bottlenecks.items():
            print(f"  {phase}: {fraction:.1%} of total time")

## Export Results

In [ ]:
# Export to CSV for sharing or further analysis
output_path = Path("results/benchmark_export.csv")
analyzer.export_csv(output_path)

print(f"Results exported to: {output_path}")

## Custom Analysis

Add your own analysis cells below.